# Session 3 — Building a Shared Repository with DagsHub and MLflow for Collaborative MLOps

**Goal:** connect MLflow's experiment tracking (Session 1) and DVC's data versioning
(Session 2) to a single **shared, hosted repository** on [DagsHub](https://dagshub.com),
so a whole team can see the same runs and pull the same data without anyone running
their own tracking server.

## What DagsHub actually provides

DagsHub hosts three things behind one Git-like URL: your **code** (a normal Git
remote), your **data** (acts as a DVC remote), and your **experiments** (acts as an
MLflow tracking server), plus a web UI to browse all three together. The MLflow and
DVC *client-side* code you already learned in Sessions 1-2 barely changes — you just
point the tracking URI / remote URL at DagsHub instead of a local folder.

## Prerequisites

This session needs a **DagsHub account** (free tier is enough) and a repository
created there — neither is available in this sandbox, so the cells below show the
complete, correct client code but are not executed here. Create a free account at
https://dagshub.com, create a repo, and get your access token from
**Settings → Tokens** before running this notebook yourself.

```bash
pip install dagshub mlflow dvc
```

In [ ]:
# Fill these in with your own DagsHub account/repo before running.
DAGSHUB_USERNAME = "your-username"
DAGSHUB_REPO = "mlops-skilling-course"
DAGSHUB_TOKEN = "your-access-token"  # Settings -> Tokens on dagshub.com

## Step 1 — Authenticate and register the repo with `dagshub`

The `dagshub` Python client handles wiring MLflow's tracking URI and any DVC remote
credentials to your repo in one call.

In [ ]:
import dagshub

dagshub.auth.add_app_token(DAGSHUB_TOKEN)
dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)

# This one call sets the MLFLOW_TRACKING_URI, MLFLOW_TRACKING_USERNAME and
# MLFLOW_TRACKING_PASSWORD environment variables for you, pointed at:
#   https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow

## Step 2 — Log runs exactly as in Session 1

This is the same `mlflow.start_run()` / `log_param` / `log_metric` / `log_model` code
from Session 1 — the only difference is *where* it's being written.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

mlflow.set_experiment("session3-shared-tracking")

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="shared_logreg"):
    model = LogisticRegression(max_iter=200).fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric("test_accuracy", acc)
    mlflow.sklearn.log_model(model, artifact_path="model")
    print(f"Logged to DagsHub. View it at:")
    print(f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow")

## Step 3 — Use DagsHub as a DVC remote

The same `.dvc` pointer-file workflow from Session 2, but `dvc push`/`dvc pull` now
talk to DagsHub instead of a local folder — anyone who clones the Git repo can
`dvc pull` the exact data version each commit points to.

In [ ]:
# Run once, inside a git+dvc initialized project (see Session 2, Steps 1-2):
#
#   dvc remote add origin https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.dvc
#   dvc remote modify origin --local auth basic
#   dvc remote modify origin --local user {DAGSHUB_USERNAME}
#   dvc remote modify origin --local password {DAGSHUB_TOKEN}
#   dvc push -r origin
print("See the commented shell commands above -- run them from a terminal in a")
print("git+dvc initialized project directory (Session 2 shows the git/dvc init steps).")

## Step 4 — A teammate's side: clone and pull

This is the collaboration payoff: a teammate gets the *exact* code, data, and view of
every experiment with two commands, no manual file-sharing.

In [ ]:
# From a teammate's machine:
#
#   git clone https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.git
#   cd {DAGSHUB_REPO}
#   dvc pull
#
# They now have the same dataset.csv Session 2 produced, and can browse every
# mlflow.start_run() from the whole team at:
#   https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}/experiments
print("Two commands (git clone + dvc pull) reproduce a teammate's exact code+data state.")

## What to try next

* Create a free DagsHub repo and re-run this notebook's cells for real — the DagsHub
  UI's experiment table is worth seeing directly, it's the same `mlflow.search_runs`
  data from Session 1 rendered as a shareable, filterable web page.
* Connect a GitHub Action (Session 10) that automatically logs a run to your DagsHub
  MLflow tracking URI on every push — turning experiment tracking from a manual step
  into a CI side effect.